# FH v3 포함 전체 브라우저 모델 새로 만들기
**기존 브라우저 변환 폴더·manifest.json·현재 PC의 모델 파일이 필요하지 않습니다.**
이전 학습에 사용한 Google Drive를 연결하고 **런타임 → 모두 실행**하세요.
Drive에 저장한 `spine_browser_model_paths.json`에서 A / 번호 / S1 학습 경로를 읽습니다.
골두는 저장된 v3 경로를 쓰거나, 기존 FH_LAT_v2의 형제 폴더 FH_LAT_v3에서 가장 최근 완료된 run을 찾습니다. 선택 경로를 출력합니다.
학습은 하지 않습니다. 원본 가중치로 6개 ONNX 그래프를 변환하고 출력값을 검증합니다. A도 새로 변환하므로 골두만 교체하는 방식보다 시간이 걸립니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 현재 웹페이지와 같은 소스 버전 사용
import io, zipfile, urllib.request, sys, subprocess
from pathlib import Path
SOURCE = Path('/content/spine_browser_converter')
SOURCE.mkdir(exist_ok=True)
url = 'https://github.com/kimtaejoon01/spine-annotator/archive/refs/heads/codex/integrated-ai-app.zip'
with urllib.request.urlopen(url) as response:
    archive = zipfile.ZipFile(io.BytesIO(response.read()))
    for item in archive.infolist():
        relative = Path(*Path(item.filename).parts[1:])
        if not relative.parts or item.is_dir(): continue
        target = (SOURCE / relative).resolve()
        if not target.is_relative_to(SOURCE.resolve()): raise ValueError('잘못된 압축 경로')
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(item))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SOURCE/'ai_server/browser-export-requirements.txt')])
sys.path.insert(0, str(SOURCE))
# 열린 런타임에서 재실행해도 이전 변환기가 남지 않도록 갱신합니다.
import importlib
importlib.invalidate_caches()
for module_name in ('ai_server.saved_export_paths', 'ai_server.s1_pair', 'ai_server.onnx_export', 'ai_server.runtime', 'ai_server.export_browser'):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])


In [ ]:
# 경로를 입력하거나 PC 파일을 업로드하지 않습니다.
from pathlib import Path
from ai_server.saved_export_paths import saved_training_paths
from ai_server.s1_pair import resolve_s1_folder
CONFIG_PATH = Path('/content/drive/MyDrive/spine_browser_model_paths.json')
paths = saved_training_paths(CONFIG_PATH)
paths['s1'] = resolve_s1_folder(paths['s1'])
for stage, path in paths.items():
    print(stage, '→', path)
print('위 Drive 학습 결과로 새 브라우저 모델을 만듭니다. 기존 변환 파일은 사용하지 않습니다.')


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import uuid
from ai_server.export_browser import export_bundle
OUTPUT = Path('/content/drive/MyDrive/spine_browser_models') / (
    'fh_v3_full_' + datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
)
export_bundle(**paths, output=OUTPUT)
print('새 모델 폴더:', OUTPUT)


In [ ]:
# 새 모델 전체를 현재 PC로 내려받습니다.
import shutil
from google.colab import files
ZIP = shutil.make_archive('/content/' + OUTPUT.name, 'zip', OUTPUT)
files.download(ZIP)


ZIP을 풀고 AI 분석 → 모델 설정 → 내 모델 폴더 선택에서 **새 폴더 전체**를 선택하세요.
앱에 **FH v3**가 표시되는지 확인하세요. 기존 학습 결과와 변환 폴더는 덮어쓰지 않습니다.
현재 PC에 기존 파일이 없어도 됩니다. 원본 학습 가중치와 저장된 경로 JSON은 연결한 Google Drive에 있어야 합니다.
